In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/PrismHire"

DATA_PATH = f"{PROJECT_PATH}/data"
OUTPUT_PATH = f"{PROJECT_PATH}/outputs"
MODEL_PATH = f"{PROJECT_PATH}/models"
GRAPH_PATH = f"{PROJECT_PATH}/graphs"

In [3]:
import zipfile

zip_path = f"{DATA_PATH}/challenge.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_PATH)

print("Dataset extracted!")

Dataset extracted!


In [4]:
import os

for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        print(os.path.join(root, file))

/content/drive/MyDrive/PrismHire/data/challenge.zip
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/.DS_Store
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/job_description.docx
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/redrob_signals_doc.docx
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/validate_submission.py
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidate_schema.json
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/submission_metadata_template.yaml
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/submission_spec.docx
/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_d

candidates.jsonl ⭐⭐⭐⭐⭐

This is almost certainly the entire candidate database.

candidate_schema.json ⭐⭐⭐⭐⭐

This tells us:

what fields exist
field types
nested structures

Think of it like a database blueprint.

job_description.docx ⭐⭐⭐⭐⭐

This is the job we are matching candidates against.

We need to understand it deeply.

redrob_signals_doc.docx ⭐⭐⭐⭐⭐

This might be the most important file.

It explains behavioral signals.

Example:

profile views
response rate
offer acceptance
search appearances

sample_submission.csv ⭐⭐⭐⭐

Tells us:

output format
column names
ranking requirements

validate_submission.py ⭐⭐⭐⭐

Tells us:

exact submission format
hidden rules
constraints

Always inspect it.

In [5]:
import pandas as pd
import json
import os

In [6]:
BASE_PATH = "/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge"

In [7]:
schema_path = f"{BASE_PATH}/candidate_schema.json"

with open(schema_path, "r") as f:
    schema = json.load(f)

print(schema)

{'$schema': 'http://json-schema.org/draft-07/schema#', 'title': 'Redrob Candidate Profile Schema', 'description': 'Schema for a single candidate profile in the Intelligent Candidate Discovery & Ranking Challenge dataset.', 'type': 'object', 'required': ['candidate_id', 'profile', 'career_history', 'education', 'skills', 'redrob_signals'], 'properties': {'candidate_id': {'type': 'string', 'pattern': '^CAND_[0-9]{7}$', 'description': 'Unique identifier for the candidate. Format: CAND_XXXXXXX (7 digits).'}, 'profile': {'type': 'object', 'required': ['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry'], 'properties': {'anonymized_name': {'type': 'string', 'description': 'Anonymized full name.'}, 'headline': {'type': 'string', 'description': 'One-line professional headline.'}, 'summary': {'type': 'string', 'description': 'Multi-sentence professional summary.'}, 'location': {'ty

In [8]:
sample_path = f"{BASE_PATH}/sample_candidates.json"

with open(sample_path, "r") as f:
    sample = json.load(f)

print(sample)

[{'candidate_id': 'CAND_0000001', 'profile': {'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | SQL, Spark, Cloud', 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", 'location': 'Toronto', 'country': 'Canada', 'years_of_experience': 6.9, 'current_title': 'Backend Engineer', 'current_company': 'Mindtree', 'current_company_size': '10001+', 'current_industry': 'IT Services'}, 'career_hist

In [9]:
jsonl_path = f"{BASE_PATH}/candidates.jsonl"

with open(jsonl_path, "r") as f:
    for i in range(5):
        line = f.readline()
        candidate = json.loads(line)

        print("="*80)
        print(candidate)

{'candidate_id': 'CAND_0000001', 'profile': {'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | SQL, Spark, Cloud', 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", 'location': 'Toronto', 'country': 'Canada', 'years_of_experience': 6.9, 'current_title': 'Backend Engineer', 'current_company': 'Mindtree', 'current_company_size': '10001+', 'current_industry': 'IT Services'}, 'career_histo

In [10]:
count = 0

with open(jsonl_path, "r") as f:
    for line in f:
        count += 1

print("Total Candidates:", count)

Total Candidates: 100000


In [11]:
import pprint
pprint.pprint(candidate)

{'candidate_id': 'CAND_0000005',
 'career_history': [{'company': 'Stark Industries',
                     'company_size': '1001-5000',
                     'description': 'Business analyst at a consulting firm, '
                                    'working primarily with retail and CPG '
                                    'clients. Conducted business diagnostics, '
                                    'process re-engineering work, and digital '
                                    'transformation strategy projects. Strong '
                                    'on stakeholder management, structured '
                                    'problem-solving, and the typical '
                                    'consulting toolkit (slide-craft, Excel '
                                    'modeling, executive communication). '
                                    'Recent project work involved AI-strategy '
                                    'advisory but my own technical depth in AI '
       

Dataset Profiling

In [12]:
import json
import pandas as pd
from tqdm import tqdm

In [13]:
candidates = []

with open(jsonl_path, "r") as f:
    for i, line in enumerate(f):
        candidates.append(json.loads(line))

        if i == 999:
            break

print(len(candidates))

1000


In [14]:
profiles = []

for c in candidates:
    p = c["profile"]

    profiles.append({
        "candidate_id": c["candidate_id"],
        "experience": p["years_of_experience"],
        "title": p["current_title"],
        "industry": p["current_industry"],
        "country": p["country"]
    })

profiles_df = pd.DataFrame(profiles)
profiles_df.head()

,candidate_id,experience,title,industry,country
0,CAND_0000001,6.9,Backend Engineer,IT Services,Canada
1,CAND_0000002,12.5,Operations Manager,IT Services,India
2,CAND_0000003,1.1,Customer Support,IT Services,USA
3,CAND_0000004,3.8,Marketing Manager,Paper Products,Australia
4,CAND_0000005,11.0,Accountant,Manufacturing,India


In [15]:
profiles_df["experience"].describe()

,experience
count,1000.00000
mean,6.89280
std,3.76027
min,1.10000
25%,3.70000
50%,6.30000
75%,9.70000
max,15.00000


In [16]:
profiles_df["industry"].value_counts().head(20)

,count
industry,
IT Services,295
Manufacturing,226
Software,214
Paper Products,73
Conglomerate,71
Food Delivery,35
Fintech,33
Consulting,19
E-commerce,12


In [17]:
skill_counts = []

for c in candidates:
    skill_counts.append(len(c["skills"]))

pd.Series(skill_counts).describe()

,0
count,1000.000000
mean,9.602000
std,3.236676
min,5.000000
25%,7.000000
50%,9.000000
75%,11.000000
max,23.000000


In [18]:
github_scores = []

for c in candidates:
    github_scores.append(
        c["redrob_signals"]["github_activity_score"]
    )

pd.Series(github_scores).describe()

,0
count,1000.00000
mean,9.42950
std,17.79883
min,-1.00000
25%,-1.00000
50%,-1.00000
75%,15.70000
max,79.40000


In [19]:
response_rates = []

for c in candidates:
    response_rates.append(
        c["redrob_signals"]["recruiter_response_rate"]
    )

pd.Series(response_rates).describe()

,0
count,1000.000000
mean,0.433280
std,0.213356
min,0.040000
25%,0.247500
50%,0.430000
75%,0.610000
max,0.910000


In [20]:
otw = []

for c in candidates:
    otw.append(
        c["redrob_signals"]["open_to_work_flag"]
    )

pd.Series(otw).value_counts()

,count
False,654
True,346


In [21]:
assessment_lengths = []

for c in candidates:
    scores = c["redrob_signals"][
        "skill_assessment_scores"
    ]

    assessment_lengths.append(len(scores))

pd.Series(assessment_lengths).describe()

,0
count,1000.000000
mean,0.361000
std,0.774131
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,5.000000


Insight 1

AI industries are rare.

Insight 2

GitHub is rare.

Insight 3

Assessments are rare.

Insight 4

Open-to-work is rare.

Insight 5

Behavior signals seem intentionally useful.

In [22]:
titles = []

for c in candidates:
    titles.append(
        c["profile"]["current_title"]
    )

pd.Series(titles).value_counts().head(30)

,count
Mechanical Engineer,70
Marketing Manager,69
Business Analyst,58
Customer Support,58
Sales Executive,57
Civil Engineer,55
HR Manager,55
Graphic Designer,55
Project Manager,54
Operations Manager,51


In [23]:
skills = []

for c in candidates:
    for s in c["skills"]:
        skills.append(s["name"])

pd.Series(skills).value_counts().head(50)

,count
Agile,144
JavaScript,143
Data Pipelines,141
Content Writing,136
AWS,134
HTML,134
Databricks,133
Accounting,132
Sales,132
Scrum,131


In [24]:
sizes = []

for c in candidates:
    sizes.append(
        c["profile"]["current_company_size"]
    )

pd.Series(sizes).value_counts()

,count
10001+,400
1001-5000,169
201-500,157
11-50,79
501-1000,76
51-200,73
5001-10000,46


In [25]:
notice = []

for c in candidates:
    notice.append(
        c["redrob_signals"]
        ["notice_period_days"]
    )

pd.Series(notice).describe()

,0
count,1000.000000
mean,87.165000
std,36.196657
min,30.000000
25%,60.000000
50%,90.000000
75%,120.000000
max,150.000000


In [26]:
saved = []

for c in candidates:
    saved.append(
        c["redrob_signals"]
        ["saved_by_recruiters_30d"]
    )

pd.Series(saved).describe()

,0
count,1000.000000
mean,7.371000
std,5.537511
min,0.000000
25%,3.000000
50%,7.000000
75%,10.250000
max,45.000000


In [27]:
completion = []

for c in candidates:
    completion.append(
        c["redrob_signals"]
        ["interview_completion_rate"]
    )

pd.Series(completion).describe()

,0
count,1000.000000
mean,0.616170
std,0.170142
min,0.300000
25%,0.480000
50%,0.610000
75%,0.760000
max,0.990000
